# Fundus domain shift: Kaggle runner

Runs the whole pipeline from the repo: APTOS 2019 (internal) -> IDRiD (external), referable DR, quality gate.

**Before running**
1. Settings -> Accelerator: **GPU T4** (or P100); Internet: **on** (to clone the repo).
2. Add input data: the **APTOS 2019 Blindness Detection** competition (accept its rules first) and an
   **IDRiD** dataset that contains the *Disease Grading* images + grading CSVs
   (e.g. `abdullahshafi315/indian-diabetic-retinopathy-image-datasetidrid`; any mirror with the original layout works).
3. Set `REPO` below to your GitHub repo URL.

Total time on a T4: roughly 10 min preprocessing + 30-60 min training.

In [ ]:
REPO = "https://github.com/<your-user>/<your-repo>.git"  # <- change me
!git clone -q $REPO /kaggle/working/repo
!pip install -q imagehash
%cd /kaggle/working/repo/files
!git log --oneline -1

In [ ]:
# Find the datasets under /kaggle/input (mount names differ between mirrors)
import glob, os
import pandas as pd

aptos_csv = next(p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
                 if "id_code" in pd.read_csv(p, nrows=0).columns)
APTOS_IMAGES = os.path.join(os.path.dirname(aptos_csv), "train_images")
idrid_csv = next(p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
                 if "Retinopathy grade" in pd.read_csv(p, nrows=0).columns.str.strip())
IDRID_ROOT = "/".join(idrid_csv.split("/")[:4])  # /kaggle/input/<dataset>
APTOS_CSV = aptos_csv
print(APTOS_CSV, APTOS_IMAGES, IDRID_ROOT, sep="\n")

## 1. Labels: IDRiD flattening, deduplication, stratified split

In [ ]:
!python prepare_idrid.py --root "$IDRID_ROOT" --out /kaggle/working/idrid_flat
!python make_csv.py --images "$APTOS_IMAGES" --labels "$APTOS_CSV" --id-col id_code --grade-col diagnosis \
    --ext .png --threshold 2 --dedup --split --out /kaggle/working/csv/aptos.csv
!python make_csv.py --images /kaggle/working/idrid_flat/images --labels /kaggle/working/idrid_flat/labels.csv \
    --id-col id --grade-col grade --ext .jpg --threshold 2 --dedup --out /kaggle/working/csv/external.csv

## 2. Preprocess: crop, pad, resize to 512, cache, quality metrics

In [ ]:
!python preprocess.py --csv /kaggle/working/csv/aptos.csv --cache-dir /kaggle/working/cache/aptos \
    --stats /kaggle/working/csv/quality_stats.json --out /kaggle/working/csv/aptos_cached.csv
!python preprocess.py --csv /kaggle/working/csv/external.csv --cache-dir /kaggle/working/cache/external \
    --stats /kaggle/working/csv/quality_stats.json --out /kaggle/working/csv/external_cached.csv

## 3. Train (model selection on validation AUROC only)

In [ ]:
!python train.py --csv /kaggle/working/csv/aptos_cached.csv --out /kaggle/working/runs/r50 --workers 4

## 4. Predict and evaluate

In [ ]:
W = "/kaggle/working"
!python predict.py --ckpt $W/runs/r50/best.pt --csv $W/csv/aptos_cached.csv --split val  --out $W/preds/val.csv  --workers 4
!python predict.py --ckpt $W/runs/r50/best.pt --csv $W/csv/aptos_cached.csv --split test --out $W/preds/test.csv --workers 4
!python predict.py --ckpt $W/runs/r50/best.pt --csv $W/csv/external_cached.csv          --out $W/preds/external.csv --workers 4
!python evaluate.py --val $W/preds/val.csv --internal $W/preds/test.csv --external $W/preds/external.csv --out $W/results

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/results/roc_and_calibration.png"))
display(Image("/kaggle/working/results/quality_gate.png"))

In [ ]:
# Lowest- and highest-quality external images: does the quality score match what you see?
import matplotlib.pyplot as plt
from PIL import Image as PILImage
ext = pd.read_csv("/kaggle/working/csv/external_cached.csv").sort_values("quality")
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (_, r) in zip(axes.flat, pd.concat([ext.head(6), ext.tail(6)]).iterrows()):
    ax.imshow(PILImage.open(r["image_path"])); ax.axis("off")
    ax.set_title(f"q={r['quality']:.2f} grade={r['grade']}", fontsize=8)
plt.suptitle("top: lowest quality, bottom: highest quality (external)"); plt.tight_layout()
plt.savefig("/kaggle/working/results/quality_examples.png", dpi=120)

In [ ]:
# Bundle what you need for the README (download from the Output tab)
!cd /kaggle/working && zip -qr results.zip results preds runs/r50/args.json csv/quality_stats.json && ls -lh results.zip